# File Practice

Apply file handling to cancellable input and text transformation. Complete the collections integration after these tasks.

These explanations and tasks retain the earlier CS1 teaching material.[^restored]


## PromptFile and Cancellation

```{index} PromptFile and Cancellation
```

Complete `StreamReader? PromptFile(string prompt)`: request a filename until it can be opened, then return the open reader. If the person enters an empty line or input ends, return `null`. The caller must check for cancellation before reading and must dispose the returned reader after use. Here `?` documents that cancellation can produce no reader; deeper nullable-type contracts are covered in Object Design.

Use `try` around the open operation. Report a missing/unreadable file and retry for `IOException` and `UnauthorizedAccessException`; do not catch unrelated programming defects. Checking `File.Exists` alone cannot guarantee the later open succeeds, because permissions or files may change between those operations.

Test a missing name followed by a valid one, immediate cancellation, and input exhaustion. Explain why the helper must not put the returned reader inside a `using` block that ends before the caller reads it.

## Copy to Upper Case

```{index} Copy to Upper Case
```

Write a program that opens an input text file, reads its lines, converts them with `ToUpperInvariant`, and writes the converted lines to a **different** output file. Retain line boundaries and empty lines. Use `using` so both streams close even if an exception occurs. Do not choose the input file as the output: opening a writer can truncate it before reading.

Create a test file containing mixed case, punctuation, and an empty line. Predict the result and compare it with the output file. Test an empty input file. Document that `WriteLine` uses the current platform's newline and may add a final newline; this is a text transformation, not a byte-for-byte copy.

### Reader ownership

```{index} Reader ownership
```

Implement PromptFile and a caller that checks cancellation and reads the first line. The answer uses a supplied input sequence so it runs without a browser prompt. Replace that sequence with Console.In in the interactive program.

In [9]:
// Exercise: Reader ownership
// Complete the task described above.
using System;
Console.WriteLine("Replace this line with your solution.");

Replace this line with your solution.


In [10]:
using System;
using System.IO;
static StreamReader? PromptFile(string prompt, TextReader input) {
    while (true) {
        Console.Write(prompt);
        string? name = input.ReadLine();
        if (string.IsNullOrEmpty(name)) return null;
        try { return new StreamReader(name); }
        catch (IOException) { Console.WriteLine("Cannot open that file. Try again."); }
        catch (UnauthorizedAccessException) { Console.WriteLine("Access denied. Try again."); }
    }
}
using StringReader choices = new StringReader("\n");
using StreamReader? reader = PromptFile("File (empty cancels): ", choices);
if (reader == null) Console.WriteLine("Cancelled.");
else Console.WriteLine(reader.ReadLine());

File (empty cancels): Cancelled.


The uppercase-copy answer uses a temporary directory so repeated runs cannot overwrite your original files. The finally block removes the demonstration files after the streams have been disposed.

In [12]:
using System;
using System.IO;
string folder = Path.Combine(Path.GetTempPath(), "uppercase-" + Guid.NewGuid());
Directory.CreateDirectory(folder);
try {
    string inputPath = Path.Combine(folder, "input.txt"), outputPath = Path.Combine(folder, "output.txt");
    File.WriteAllText(inputPath, "Hello, World!\n\nOrder 42\n");
    using (StreamReader input = new StreamReader(inputPath))
    using (StreamWriter output = new StreamWriter(outputPath)) {
        string? line;
        while ((line = input.ReadLine()) != null) output.WriteLine(line.ToUpperInvariant());
    }
    Console.Write(File.ReadAllText(outputPath));
}
finally { Directory.Delete(folder, true); }

HELLO, WORLD!

ORDER 42


```{rubric} Footnotes
```

[^restored]: Restored from the Files lab. The cancellation and uppercase-copy tasks complement the current record-processing and Grade Files assignments.